# Activation steering for GYAFC formality

This notebook tests the causal question left open by the representation experiment:

> Does adding the discovered GYAFC formalization direction to a decoder layer cause controllable changes in generated text?

The notebook recomputes three directions using the same deterministic extraction split, verifies the intervention numerically, runs an alpha sweep on held-out sentences, and compares generated outputs with unsteered and prompt-only baselines.

The three candidates are:

- Primary: mean-pooled direction after decoder layer 21.
- Alternative: final-token direction after decoder layer 4.
- Layer-11 comparison: mean-pooled direction after decoder layer 11. It was originally selected as a weak-geometry control, but the generation results may establish it as a genuine steering candidate.

## Experimental logic

The representation experiment established correlation: formal and informal sentences separate along particular directions. Steering tests causation by modifying the residual activation during generation:

h_steered = h + alpha × v_formal

Negative alpha should move against the formalization direction, zero should reproduce the normal model, and positive alpha should move toward the direction. A convincing result should be graded rather than appearing only at one extreme alpha, should preserve meaning over a useful range, and should be stronger at the validated layer than at the negative-control layer.

In [1]:
from contextlib import contextmanager, nullcontext
from difflib import SequenceMatcher
from pathlib import Path
import gc
import math
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
torch.manual_seed(42)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Machine:", platform.machine())
print("Python executable:", sys.executable)

Python: 3.11.0
PyTorch: 2.12.1
Transformers: 5.14.1
Machine: arm64
Python executable: /usr/local/bin/python3


## 1. Load the same model

The extraction and intervention model must be identical. The Mac uses MPS with float16 when available.

In [2]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

MODEL_DTYPE = torch.float16 if device.type in {"mps", "cuda"} else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=MODEL_DTYPE)
model = model.to(device)
model.eval()

config = model.config
decoder_layers = model.model.layers
print("Device:", device)
print("Model dtype:", next(model.parameters()).dtype)
print("Decoder layers:", config.num_hidden_layers)
print("Hidden size:", config.hidden_size)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Device: mps
Model dtype: torch.float16
Decoder layers: 24
Hidden size: 896


## 2. Recreate the extraction and held-out split

This reproduces the sampling policy used in the large representation experiment: exact copies are excluded, each domain contributes equally, and no held-out pair contributes to a steering direction.

In [3]:
DATA_ROOT = Path("/Users/nandanprince/Downloads/GYAFC_Corpus")
DOMAIN_DIRECTORIES = {
    "entertainment_music": "Entertainment_Music",
    "family_relationships": "Family_Relationships",
}
EXPERIMENT_SEED = 42
EXTRACTION_PAIRS_PER_DOMAIN = 4_000
HELDOUT_PAIRS_PER_DOMAIN = 1_000


def load_training_pairs() -> pd.DataFrame:
    frames = []
    for domain, directory in DOMAIN_DIRECTORIES.items():
        train_directory = DATA_ROOT / directory / "train"
        informal = (train_directory / "informal").read_text(encoding="utf-8").splitlines()
        formal = (train_directory / "formal").read_text(encoding="utf-8").splitlines()
        assert len(informal) == len(formal)
        frames.append(
            pd.DataFrame(
                {
                    "domain": domain,
                    "domain_row": np.arange(len(informal)),
                    "informal": informal,
                    "formal": formal,
                }
            )
        )

    frame = pd.concat(frames, ignore_index=True)
    informal_normalized = frame["informal"].str.strip().str.lower().str.replace(
        r"\s+", " ", regex=True
    )
    formal_normalized = frame["formal"].str.strip().str.lower().str.replace(
        r"\s+", " ", regex=True
    )
    frame["exact_copy"] = informal_normalized == formal_normalized
    return frame


all_pairs = load_training_pairs()
eligible_pairs = all_pairs.loc[~all_pairs["exact_copy"]].copy()
split_rng = np.random.default_rng(EXPERIMENT_SEED)
extraction_parts = []
heldout_parts = []

for domain in DOMAIN_DIRECTORIES:
    domain_frame = eligible_pairs.loc[eligible_pairs["domain"] == domain]
    required = EXTRACTION_PAIRS_PER_DOMAIN + HELDOUT_PAIRS_PER_DOMAIN
    positions = split_rng.permutation(len(domain_frame))
    extraction_parts.append(domain_frame.iloc[positions[:EXTRACTION_PAIRS_PER_DOMAIN]])
    heldout_parts.append(
        domain_frame.iloc[positions[EXTRACTION_PAIRS_PER_DOMAIN:required]]
    )

extraction_pairs = (
    pd.concat(extraction_parts)
    .sample(frac=1, random_state=EXPERIMENT_SEED)
    .reset_index(drop=True)
)
heldout_pairs = (
    pd.concat(heldout_parts)
    .sample(frac=1, random_state=EXPERIMENT_SEED + 1)
    .reset_index(drop=True)
)

assert not set(zip(extraction_pairs["domain"], extraction_pairs["domain_row"])) & set(
    zip(heldout_pairs["domain"], heldout_pairs["domain_row"])
)
display(
    pd.DataFrame(
        {
            "extraction": extraction_pairs.groupby("domain").size(),
            "heldout": heldout_pairs.groupby("domain").size(),
        }
    )
)

,extraction,heldout
domain,,
entertainment_music,4000,1000
family_relationships,4000,1000


## 3. Recompute only the selected directions

Forward hooks capture only the three validated layers, avoiding storage of every hidden state. The primary and control candidates use mean pooling; the alternative uses the final non-padding token.

In [4]:
CANDIDATES = {
    "primary_layer21_mean": {"layer_index": 21, "pooling": "mean"},
    "alternative_layer4_final": {"layer_index": 4, "pooling": "final_token"},
    "negative_layer11_mean": {"layer_index": 11, "pooling": "mean"},
}
DIRECTION_BATCH_SIZE = 32
DIRECTION_MAX_LENGTH = 128


def mean_pool(hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    expanded_mask = attention_mask.unsqueeze(-1).to(hidden_state.dtype)
    return (hidden_state * expanded_mask).sum(dim=1) / expanded_mask.sum(
        dim=1
    ).clamp_min(1)


def final_token_pool(
    hidden_state: torch.Tensor, attention_mask: torch.Tensor
) -> torch.Tensor:
    final_positions = attention_mask.sum(dim=1) - 1
    batch_positions = torch.arange(hidden_state.shape[0], device=hidden_state.device)
    return hidden_state[batch_positions, final_positions]


captured_layers = {}


def make_capture_hook(candidate_name):
    def capture(module, module_inputs, module_output):
        residual = module_output[0] if isinstance(module_output, tuple) else module_output
        captured_layers[candidate_name] = residual
    return capture


direction_sums = {
    name: torch.zeros(config.hidden_size, dtype=torch.float32)
    for name in CANDIDATES
}
capture_handles = [
    decoder_layers[settings["layer_index"]].register_forward_hook(
        make_capture_hook(name)
    )
    for name, settings in CANDIDATES.items()
]

started = time.perf_counter()
number_of_batches = math.ceil(len(extraction_pairs) / DIRECTION_BATCH_SIZE)
try:
    for batch_number, start in enumerate(
        range(0, len(extraction_pairs), DIRECTION_BATCH_SIZE), start=1
    ):
        stop = min(start + DIRECTION_BATCH_SIZE, len(extraction_pairs))
        batch = extraction_pairs.iloc[start:stop]
        pair_count = len(batch)
        texts = batch["informal"].tolist() + batch["formal"].tolist()
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=DIRECTION_MAX_LENGTH,
            return_tensors="pt",
        )
        batch_inputs = {name: tensor.to(device) for name, tensor in encoded.items()}
        captured_layers.clear()

        with torch.inference_mode():
            model.model(
                **batch_inputs,
                use_cache=False,
                return_dict=True,
            )

        for name, settings in CANDIDATES.items():
            hidden = captured_layers[name]
            if settings["pooling"] == "mean":
                representations = mean_pool(hidden, batch_inputs["attention_mask"])
            else:
                representations = final_token_pool(
                    hidden, batch_inputs["attention_mask"]
                )
            deltas = representations[pair_count:] - representations[:pair_count]
            direction_sums[name] += deltas.float().sum(dim=0).cpu()

        if batch_number == 1 or batch_number % 25 == 0 or batch_number == number_of_batches:
            print(
                f"batch {batch_number}/{number_of_batches}: {stop:,}/{len(extraction_pairs):,} "
                f"pairs, {time.perf_counter() - started:.1f}s"
            )
finally:
    for handle in capture_handles:
        handle.remove()
    captured_layers.clear()

raw_directions = {
    name: direction_sum / len(extraction_pairs)
    for name, direction_sum in direction_sums.items()
}
unit_directions = {
    name: vector / vector.norm().clamp_min(1e-12)
    for name, vector in raw_directions.items()
}

direction_summary = pd.DataFrame(
    [
        {
            "candidate": name,
            **settings,
            "raw_direction_norm": raw_directions[name].norm().item(),
        }
        for name, settings in CANDIDATES.items()
    ]
)
display(direction_summary.round(4))

ARTIFACT_DIRECTORY = Path("/Users/nandanprince/Desktop/NLP_project/SecondThought/steering_experimentation/artifacts")
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)
DIRECTION_PATH = ARTIFACT_DIRECTORY / "qwen2_5_0_5b_gyafc_formality_directions.pt"
torch.save(
    {
        "model_id": MODEL_ID,
        "seed": EXPERIMENT_SEED,
        "extraction_pairs": len(extraction_pairs),
        "candidates": CANDIDATES,
        "raw_directions": raw_directions,
        "unit_directions": unit_directions,
    },
    DIRECTION_PATH,
)
print("Saved directions to:", DIRECTION_PATH)

batch 1/250: 32/8,000 pairs, 0.7s
batch 25/250: 800/8,000 pairs, 9.1s
batch 50/250: 1,600/8,000 pairs, 17.5s
batch 75/250: 2,400/8,000 pairs, 26.5s
batch 100/250: 3,200/8,000 pairs, 35.3s
batch 125/250: 4,000/8,000 pairs, 44.0s
batch 150/250: 4,800/8,000 pairs, 52.2s
batch 175/250: 5,600/8,000 pairs, 61.0s
batch 200/250: 6,400/8,000 pairs, 69.0s
batch 225/250: 7,200/8,000 pairs, 76.9s
batch 250/250: 8,000/8,000 pairs, 85.1s


,candidate,layer_index,pooling,raw_direction_norm
0,primary_layer21_mean,21,mean,5.6086
1,alternative_layer4_final,4,final_token,2.5102
2,negative_layer11_mean,11,mean,3.9048


Saved directions to: /Users/nandanprince/Desktop/NLP_project/SecondThought/steering_experimentation/artifacts/qwen2_5_0_5b_gyafc_formality_directions.pt


## 4. Define and verify the steering hook

The hook adds the raw mean difference vector to every token position returned by the selected decoder layer. Alpha is therefore dimensionless: alpha 1 adds one average formal-minus-informal displacement. The tuple structure returned by the decoder layer is preserved.

In [5]:
def make_steering_hook(vector: torch.Tensor, alpha: float, audit=None):
    def steering_hook(module, module_inputs, module_output):
        is_tuple = isinstance(module_output, tuple)
        residual = module_output[0] if is_tuple else module_output
        shift = (alpha * vector).to(device=residual.device, dtype=residual.dtype)
        modified = residual + shift.view(1, 1, -1)

        if audit is not None and "before" not in audit:
            audit["before"] = residual[:, -1, :].detach().float().cpu()
            audit["after"] = modified[:, -1, :].detach().float().cpu()
            audit["expected_shift"] = shift.detach().float().cpu()

        if is_tuple:
            return (modified, *module_output[1:])
        return modified

    return steering_hook


@contextmanager
def steering_context(candidate_name: str, alpha: float, audit=None):
    settings = CANDIDATES[candidate_name]
    handle = decoder_layers[settings["layer_index"]].register_forward_hook(
        make_steering_hook(raw_directions[candidate_name], alpha, audit=audit)
    )
    try:
        yield
    finally:
        handle.remove()


verification_sentence = heldout_pairs.iloc[0]["informal"]
verification_inputs = tokenizer(
    verification_sentence, return_tensors="pt"
).to(device)
verification_audit = {}
with steering_context("primary_layer21_mean", alpha=1.0, audit=verification_audit):
    with torch.inference_mode():
        model(
            **verification_inputs,
            use_cache=False,
            return_dict=True,
        )

observed_shift = verification_audit["after"] - verification_audit["before"]
expected_shift = verification_audit["expected_shift"].unsqueeze(0)
maximum_shift_error = (observed_shift - expected_shift).abs().max().item()
observed_shift_cosine = F.cosine_similarity(
    observed_shift.mean(dim=0),
    verification_audit["expected_shift"],
    dim=0,
).item()

print("Direction norm:", raw_directions["primary_layer21_mean"].norm().item())
print("Observed shift norm:", observed_shift.mean(dim=0).norm().item())
print("Maximum shift error:", maximum_shift_error)
print("Observed/expected shift cosine:", observed_shift_cosine)
assert maximum_shift_error < 0.05


Direction norm: 5.608584880828857
Observed shift norm: 5.610051155090332
Maximum shift error: 0.00579833984375
Observed/expected shift cosine: 0.9999983906745911


## 5. Generate held-out rewrites across steering strengths

Every enabled candidate uses the same meaning-preserving neutral prompt and the same alpha sweep. Alpha zero is the shared unsteered baseline. Generation is greedy so repeated runs are deterministic. Candidate enable flags let each layer be run independently.

In [6]:
NEUTRAL_SYSTEM_PROMPT = (
    "Rewrite the user's sentence while preserving its meaning. "
    "Return only the rewritten sentence."
)
FORMAL_SYSTEM_PROMPT = (
    "Rewrite the user's sentence in formal English while preserving its meaning. "
    "Return only the rewritten sentence."
)


def generate_rewrite(
    sentence: str,
    system_prompt: str,
    candidate_name: str | None = None,
    alpha: float = 0.0,
    max_new_tokens: int = 64,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sentence},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    context = (
        steering_context(candidate_name, alpha)
        if candidate_name is not None and alpha != 0
        else nullcontext()
    )
    with context:
        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    new_ids = generated_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


In [9]:
# Toggle any candidate independently. The same prompt, examples, decoding
# settings, and alpha values are used for every enabled candidate.
RUN_CANDIDATES = {
    "primary_layer21_mean": False,
    "alternative_layer4_final": False,
    "negative_layer11_mean": True,
}
EVALUATION_EXAMPLES_PER_DOMAIN = 10
ALPHAS = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]
GENERATION_SYSTEM_PROMPT = NEUTRAL_SYSTEM_PROMPT

candidates_to_run = [
    name for name, should_run in RUN_CANDIDATES.items() if should_run
]
unknown_candidates = set(candidates_to_run) - set(CANDIDATES)
if unknown_candidates:
    raise ValueError(f"Unknown candidates: {sorted(unknown_candidates)}")
if not candidates_to_run:
    raise ValueError("Enable at least one entry in RUN_CANDIDATES.")

steering_examples = (
    heldout_pairs.groupby("domain", group_keys=False)
    .head(EVALUATION_EXAMPLES_PER_DOMAIN)
    .reset_index(drop=True)
)

total_generations = (
    len(candidates_to_run) * len(steering_examples) * len(ALPHAS)
)
generation_number = 0
started = time.perf_counter()
steering_outputs_by_candidate = {}

for candidate_name in candidates_to_run:
    candidate_rows = []
    print(f"Starting candidate: {candidate_name}")

    for example_id, row in steering_examples.iterrows():
        for alpha in ALPHAS:
            generation_number += 1
            candidate_rows.append(
                {
                    "example_id": example_id,
                    "domain": row["domain"],
                    "informal": row["informal"],
                    "formal_reference": row["formal"],
                    "condition": candidate_name,
                    "alpha": alpha,
                    "generated": generate_rewrite(
                        row["informal"],
                        GENERATION_SYSTEM_PROMPT,
                        candidate_name=candidate_name,
                        alpha=alpha,
                    ),
                }
            )

        print(
            f"{candidate_name}: example {example_id + 1}/{len(steering_examples)} "
            f"({generation_number}/{total_generations} total generations, "
            f"{time.perf_counter() - started:.1f}s)"
        )

    candidate_outputs = pd.DataFrame(candidate_rows)
    steering_outputs_by_candidate[candidate_name] = candidate_outputs
    candidate_output_path = (
        ARTIFACT_DIRECTORY / f"activation_steering_{candidate_name}.csv"
    )
    candidate_outputs.to_csv(candidate_output_path, index=False)
    print(f"Saved {candidate_name} outputs to: {candidate_output_path}")
    display(
        candidate_outputs[
            ["example_id", "condition", "alpha", "informal", "generated"]
        ]
    )

steering_outputs = pd.concat(
    steering_outputs_by_candidate.values(), ignore_index=True
)
OUTPUT_PATH = ARTIFACT_DIRECTORY / "activation_steering_selected_candidates.csv"
steering_outputs.to_csv(OUTPUT_PATH, index=False)
print("Saved combined selected-candidate outputs to:", OUTPUT_PATH)

Starting candidate: negative_layer11_mean
negative_layer11_mean: example 1/20 (7/140 total generations, 2.4s)
negative_layer11_mean: example 2/20 (14/140 total generations, 4.4s)
negative_layer11_mean: example 3/20 (21/140 total generations, 5.1s)
negative_layer11_mean: example 4/20 (28/140 total generations, 6.8s)
negative_layer11_mean: example 5/20 (35/140 total generations, 8.0s)
negative_layer11_mean: example 6/20 (42/140 total generations, 9.0s)
negative_layer11_mean: example 7/20 (49/140 total generations, 10.0s)
negative_layer11_mean: example 8/20 (56/140 total generations, 13.6s)
negative_layer11_mean: example 9/20 (63/140 total generations, 16.2s)
negative_layer11_mean: example 10/20 (70/140 total generations, 18.3s)
negative_layer11_mean: example 11/20 (77/140 total generations, 19.5s)
negative_layer11_mean: example 12/20 (84/140 total generations, 20.3s)
negative_layer11_mean: example 13/20 (91/140 total generations, 22.7s)
negative_layer11_mean: example 14/20 (98/140 total 

,example_id,condition,alpha,informal,generated
0,0,negative_layer11_mean,-2.0,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly love her, if not as a lover then surely as a friend."
1,0,negative_layer11_mean,-1.0,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly love her, if not as a lover then surely as a friend."
2,0,negative_layer11_mean,-0.5,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly love her, but more likely as a friend."
3,0,negative_layer11_mean,0.0,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly love her, but more so as a friend than a lover."
4,0,negative_layer11_mean,0.5,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly adore her, whether she is your lover or just a friend."
5,0,negative_layer11_mean,1.0,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly admire her, but more than that, you also consider her to be a friend."
6,0,negative_layer11_mean,2.0,"Well you obviously love her, if not as a lover then surely as a friend.","You clearly admire her both as a lover and a friend, but this admiration is more than superficial; it goes beyond me..."
7,1,negative_layer11_mean,-2.0,Like the word slacks....wonder if a guy can wear a just a slack?,like the word slacks... wonder if a guy can wear a just a slack?
8,1,negative_layer11_mean,-1.0,Like the word slacks....wonder if a guy can wear a just a slack?,"Like the word ""slacks,"" wonder if a guy could wear a just a slack?"
9,1,negative_layer11_mean,-0.5,Like the word slacks....wonder if a guy can wear a just a slack?,"Like the word ""slacks,"" wonder if a guy could wear a ""just a slack""?"


Saved combined selected-candidate outputs to: /Users/nandanprince/Desktop/NLP_project/SecondThought/steering_experimentation/artifacts/activation_steering_selected_candidates.csv


## 6. Inspect lightweight trade-off indicators

These surface measurements are diagnostic, not definitive formality or semantic metrics. Character similarity can penalize valid paraphrases and reward exact copies. Informal-marker counts cover only a small hand-written vocabulary. The generated examples must still be read.

In [10]:
WORD_RE = re.compile(r"\b[\w']+\b")
CONTRACTION_RE = re.compile(
    r"\b(?:ain't|aren't|can't|couldn't|didn't|doesn't|don't|hadn't|hasn't|haven't|"
    r"he's|i'd|i'll|i'm|i've|isn't|it's|shouldn't|that's|they're|wasn't|weren't|"
    r"what's|won't|wouldn't|you're|you've)\b",
    flags=re.IGNORECASE,
)
SLANG_RE = re.compile(
    r"\b(?:u|ur|r|lol|omg|idk|imo|btw|thx|thanx|wanna|gonna|gotta|cuz|coz|kinda)\b",
    flags=re.IGNORECASE,
)


def output_diagnostics(source: str, generated: str) -> dict[str, float]:
    source_words = WORD_RE.findall(source.lower())
    generated_words = WORD_RE.findall(generated.lower())
    source_word_set = set(source_words)
    generated_word_set = set(generated_words)
    union = source_word_set | generated_word_set
    return {
        "word_count": len(generated_words),
        "length_ratio": (len(generated_words) + 1) / (len(source_words) + 1),
        "contractions": len(CONTRACTION_RE.findall(generated)),
        "slang_tokens": len(SLANG_RE.findall(generated)),
        "exclamation_marks": generated.count("!"),
        "terminal_punctuation": float(generated.rstrip().endswith((".", "!", "?"))),
        "character_similarity_to_input": SequenceMatcher(
            None, source.lower(), generated.lower(), autojunk=False
        ).ratio(),
        "word_set_jaccard_to_input": (
            len(source_word_set & generated_word_set) / len(union) if union else 1.0
        ),
        "exact_copy": float(source.strip().lower() == generated.strip().lower()),
    }


diagnostics = pd.DataFrame(
    [
        output_diagnostics(row.informal, row.generated)
        for row in steering_outputs.itertuples()
    ]
)
steering_outputs = pd.concat([steering_outputs, diagnostics], axis=1)
steering_outputs["informal_marker_count"] = (
    steering_outputs["contractions"]
    + steering_outputs["slang_tokens"]
    + steering_outputs["exclamation_marks"]
)

candidate_tradeoffs = (
    steering_outputs.groupby(["condition", "alpha"], dropna=False)
    .agg(
        informal_marker_count=("informal_marker_count", "mean"),
        terminal_punctuation_rate=("terminal_punctuation", "mean"),
        mean_word_count=("word_count", "mean"),
        mean_length_ratio=("length_ratio", "mean"),
        character_similarity_to_input=("character_similarity_to_input", "mean"),
        word_set_jaccard_to_input=("word_set_jaccard_to_input", "mean"),
        exact_copy_rate=("exact_copy", "mean"),
    )
)
print("Alpha trajectory for every enabled candidate:")
display(candidate_tradeoffs.round(4))

for candidate_name in candidates_to_run:
    candidate_path = (
        ARTIFACT_DIRECTORY / f"activation_steering_{candidate_name}.csv"
    )
    candidate_frame = steering_outputs.loc[
        steering_outputs["condition"] == candidate_name
    ]
    candidate_frame.to_csv(candidate_path, index=False)
steering_outputs.to_csv(OUTPUT_PATH, index=False)
print("Updated candidate CSVs with surface diagnostics.")

Alpha trajectory for every enabled candidate:


informal_marker_count  terminal_punctuation_rate  \
condition             alpha                                                     
negative_layer11_mean -2.0                    0.35                       0.35   
                      -1.0                    0.20                       0.70   
                      -0.5                    0.15                       0.85   
                       0.0                    0.15                       0.85   
                       0.5                    0.25                       0.90   
                       1.0                    0.15                       0.95   
                       2.0                    0.00                       1.00   

                             mean_word_count  mean_length_ratio  \
condition             alpha                                       
negative_layer11_mean -2.0             10.45             0.9127   
                      -1.0              9.90             0.8724   
                      -0.5             10.25             0.9084   
                       0.0             11.00             0.9563   
                       0.5             10.85             0.9605   
                       1.0             12.95             1.1108   
                       2.0             12.20             1.0747   

                             character_similarity_to_input  \
condition             alpha                                  
negative_layer11_mean -2.0                          0.8824   
                      -1.0                          0.8006   
                      -0.5                          0.7558   
                       0.0                          0.7285   
                       0.5                          0.6534   
                       1.0                          0.5901   
                       2.0                          0.5150   

                             word_set_jaccard_to_input  exact_copy_rate  
condition             alpha                                              
negative_layer11_mean -2.0                      0.7437             0.05  
                      -1.0                      0.6451             0.00  
                      -0.5                      0.5715             0.00  
                       0.0                      0.5356             0.00  
                       0.5                      0.4179             0.00  
                       1.0                      0.3782             0.00  
                       2.0                      0.3206             0.00

Updated candidate CSVs with surface diagnostics.


## 7. Re-encode outputs and measure their position along the direction

Each generated sentence is re-encoded and projected onto the same candidate direction used to steer it. This is a useful internal check but is circular: the intervention and measurement use the same direction. Projection values from different layers are not directly comparable. Human inspection and independent style/semantic metrics remain necessary.

In [11]:
def score_texts_on_candidate_direction(
    texts: list[str], candidate_name: str, batch_size: int = 32
) -> np.ndarray:
    settings = CANDIDATES[candidate_name]
    scores = []
    captured = {}

    def capture(module, module_inputs, module_output):
        captured["hidden"] = (
            module_output[0] if isinstance(module_output, tuple) else module_output
        )

    handle = decoder_layers[settings["layer_index"]].register_forward_hook(capture)
    try:
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            )
            inputs = {name: tensor.to(device) for name, tensor in encoded.items()}
            captured.clear()
            with torch.inference_mode():
                model.model(**inputs, use_cache=False, return_dict=True)
            hidden = captured["hidden"]
            if settings["pooling"] == "mean":
                representations = mean_pool(hidden, inputs["attention_mask"])
            else:
                representations = final_token_pool(hidden, inputs["attention_mask"])
            direction = unit_directions[candidate_name].to(
                device=representations.device, dtype=representations.dtype
            )
            scores.extend((representations @ direction).float().cpu().tolist())
    finally:
        handle.remove()
    return np.asarray(scores)


projection_baseline_rows = []
steering_outputs["own_direction_projection"] = np.nan

for candidate_name in candidates_to_run:
    informal_projections = score_texts_on_candidate_direction(
        steering_examples["informal"].tolist(), candidate_name
    )
    formal_reference_projections = score_texts_on_candidate_direction(
        steering_examples["formal"].tolist(), candidate_name
    )
    projection_baseline_rows.append(
        {
            "candidate": candidate_name,
            "mean_informal_projection": informal_projections.mean(),
            "mean_formal_reference_projection": formal_reference_projections.mean(),
            "mean_formal_minus_informal_gap": (
                formal_reference_projections - informal_projections
            ).mean(),
            "formal_above_informal_rate": (
                formal_reference_projections > informal_projections
            ).mean(),
        }
    )

    candidate_mask = steering_outputs["condition"] == candidate_name
    steering_outputs.loc[candidate_mask, "own_direction_projection"] = (
        score_texts_on_candidate_direction(
            steering_outputs.loc[candidate_mask, "generated"].tolist(),
            candidate_name,
        )
    )

projection_baselines = pd.DataFrame(projection_baseline_rows).set_index("candidate")
print("Projection anchors for each enabled candidate:")
display(projection_baselines.round(4))

projection_trajectories = (
    steering_outputs.groupby(["condition", "alpha"])
    .agg(
        mean_output_projection=("own_direction_projection", "mean"),
        projection_std=("own_direction_projection", "std"),
        character_similarity_to_input=("character_similarity_to_input", "mean"),
        informal_marker_count=("informal_marker_count", "mean"),
    )
)
print("Projection trajectory for every enabled candidate:")
display(projection_trajectories.round(4))

for candidate_name in candidates_to_run:
    candidate_path = (
        ARTIFACT_DIRECTORY / f"activation_steering_{candidate_name}.csv"
    )
    steering_outputs.loc[
        steering_outputs["condition"] == candidate_name
    ].to_csv(candidate_path, index=False)
steering_outputs.to_csv(OUTPUT_PATH, index=False)
print("Updated candidate and combined CSVs with projection scores.")

/var/folders/nf/h6pl836x6_39vqhbwflhx77w0000gn/T/ipykernel_85267/4199796476.py:36: UserWarning: MPS mm implementation has a known issue with this shape, dtype and slice. Dispatching to metal implementation instead. This may impact performance. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/mps/operations/LinearAlgebra.mm:322.)
  scores.extend((representations @ direction).float().cpu().tolist())


Projection anchors for each enabled candidate:


,mean_informal_projection,mean_formal_reference_projection,mean_formal_minus_informal_gap,formal_above_informal_rate
candidate,,,,
negative_layer11_mean,107.6594,105.4406,-2.2188,0.5


Projection trajectory for every enabled candidate:


mean_output_projection  projection_std  \
condition             alpha                                           
negative_layer11_mean -2.0                 134.3656         61.7606   
                      -1.0                 130.1844         54.5758   
                      -0.5                 122.5656         49.1948   
                       0.0                 122.2734         56.5393   
                       0.5                 118.9047         45.3748   
                       1.0                 110.5547         49.9010   
                       2.0                 108.9469         42.7903   

                             character_similarity_to_input  \
condition             alpha                                  
negative_layer11_mean -2.0                          0.8824   
                      -1.0                          0.8006   
                      -0.5                          0.7558   
                       0.0                          0.7285   
                       0.5                          0.6534   
                       1.0                          0.5901   
                       2.0                          0.5150   

                             informal_marker_count  
condition             alpha                         
negative_layer11_mean -2.0                    0.35  
                      -1.0                    0.20  
                      -0.5                    0.15  
                       0.0                    0.15  
                       0.5                    0.25  
                       1.0                    0.15  
                       2.0                    0.00

Updated candidate and combined CSVs with projection scores.


## 8. Interpretation checklist

Evidence for causal and controllable steering would include:

1. Outputs change progressively as alpha moves from negative to positive.
2. Positive alpha produces more formal language than alpha zero, while negative alpha moves in the opposite direction.
3. Candidate layers show distinguishable, reproducible trajectories under matched prompts and alpha values; a separate random or shuffled direction should be added later as the true negative control.
4. A useful alpha range improves style without large losses in meaning or fluency.
5. Large alpha eventually reveals predictable failure modes rather than merely random output changes.

Do not conclude success solely because output projection increases. Read every trajectory by example and later add independent formality and semantic-similarity evaluators. If the primary direction changes spelling, punctuation, length, or topic without consistently improving formality, the geometric direction is real but its causal interpretation is narrower than intended.

## 9. Results report: layer 11 mean-pooled steering

### What was tested

This run applies the mean-pooled formal-minus-informal direction at decoder layer 11. It contains 20 held-out inputs (10 from each domain) and seven alpha values from -2 to +2, for 140 deterministic generations. The prompt and decoding settings are fixed, so alpha is the intended experimental variable. Alpha zero is still a neutral *rewrite* prompt; it is not an unprompted model output or a copy baseline.

The earlier hook check showed that the intervention code adds the requested vector with cosine similarity 0.999998 and very small coordinate error. That check used the layer-21 candidate, however, so it validates the shared hook mechanism rather than independently rechecking the layer-11 vector.

### Aggregate alpha response

| Alpha | Informal markers | Terminal punctuation | Mean length ratio | Character similarity | Word Jaccard | Mean direction projection |
|---:|---:|---:|---:|---:|---:|---:|
| -2.0 | 0.35 | 35% | 0.913 | 0.882 | 0.744 | 134.37 |
| -1.0 | 0.20 | 70% | 0.872 | 0.801 | 0.645 | 130.18 |
| -0.5 | 0.15 | 85% | 0.908 | 0.756 | 0.572 | 122.57 |
| 0.0 | 0.15 | 85% | 0.956 | 0.729 | 0.536 | 122.27 |
| +0.5 | 0.25 | 90% | 0.961 | 0.653 | 0.418 | 118.90 |
| +1.0 | 0.15 | 95% | 1.111 | 0.590 | 0.378 | 110.55 |
| +2.0 | 0.00 | 100% | 1.075 | 0.515 | 0.321 | 108.95 |

There is a strong, ordered intervention effect. Across the seven alpha settings, terminal punctuation rises almost monotonically, informal-marker counts generally fall, and outputs become longer at the strongest positive settings. More importantly, both character similarity and word-set overlap decrease at every successive alpha. Their Spearman correlation with alpha is -1.0 over these seven aggregate points. This is compelling evidence that alpha controls the strength of a reproducible rewrite behavior rather than producing unrelated random variation. The marker heuristic is coarse—the increase at alpha +0.5 is a reminder not to treat any single surface metric as a formality score.

### Qualitative interpretation

The direction has a recognizable formalization effect. Positive alpha expands contractions, regularizes capitalization and punctuation, and favors more formal vocabulary. For example, `its really hard when u want to know what happened` progresses through `it is very difficult...` and then `It is extremely challenging to ascertain what occurred.` Negative alpha tends to preserve the wording and informal surface form more closely.

The same trajectory also exposes a loss-of-meaning frontier. At +1 and especially +2, the model frequently substitutes, deletes, or invents content: `slacks` becomes `shorts`; `What if he left his wife and family for you?` becomes a question about leaving for `someone else`; a ranked list loses or reorders items; and `Once a snake, always a snake` is distorted at +2. Some outputs also become unnecessarily abstract or verbose. Thus larger positive alpha is not simply 'more formal.' It increasingly controls lexical sophistication, elaboration, and rewrite intensity, with semantic faithfulness as the cost.

On this small evaluation, approximately +0.5 appears to be the most conservative useful setting, while +1 is visibly stronger but already risky. Alpha +2 is primarily a stress test, not a good operating point. This is a provisional judgment because the current metrics do not directly measure formality or semantic equivalence.

### Important contradiction in the projection test

The re-encoded outputs do **not** move in the expected positive direction. Mean projection decreases monotonically from 134.37 at alpha -2 to 108.95 at alpha +2. On these same 20 examples, the formal references have a lower mean projection than the informal inputs (105.44 versus 107.66), a mean formal-minus-informal gap of -2.22, and only 50% of formal references score above their paired informal inputs. This agrees with the earlier held-out geometry result for this candidate: about 52.85% pair-ordering accuracy and a small effect size of 0.084.

This does not invalidate the causal alpha effect. The intervention is applied inside a prompted generation trajectory, whereas the projection test re-encodes the completed text as a fresh input; nonlinear downstream processing and the distribution change can make those measurements disagree. It does mean that the evidence is currently stronger for a **causal formalization-like rewrite control** than for a clean, globally valid one-dimensional formality representation at layer 11. Layer 11 should therefore be promoted from 'negative control' to an interesting causal candidate, but not yet called a validated formality direction.

### Conclusion

Yes, alpha is doing meaningful and directional work. Positive layer-11 steering usually makes the responses look more formal, and negative steering keeps them closer to the informal source. Your observation about alpha +2 is supported quantitatively: it gives the cleanest surface form but the lowest source similarity and word overlap, alongside clear qualitative meaning errors. The experiment establishes controllability, while leaving the exact concept being controlled unresolved.

### Next steps

1. **Measure the actual objective independently.** Score every output for formality with a held-out classifier or human ratings, and score semantic preservation with a sentence-embedding metric plus targeted human checks. The evaluator must not reuse the steering direction.
2. **Run matched baselines.** Compare layer 11 against alpha zero, the layer-4 final-token and layer-21 mean candidates, a random norm-matched vector, and several shuffled-label directions. This will show whether the smooth alpha response is specific to the learned formality vector.
3. **Use a denser safe-range sweep.** Test approximately 0.0 to 1.0 in 0.1 or 0.25 steps. Select alpha on a validation set using a joint criterion: maximum formality improvement subject to a semantic-similarity or human-faithfulness threshold. Do not tune on these 20 displayed examples.
4. **Increase and stratify the held-out set.** Evaluate many more examples from both domains and report paired confidence intervals or bootstrap intervals. Break results down by contractions, slang, sentence length, questions, named entities, numbers, and lists, because those are visible failure modes.
5. **Diagnose the projection reversal.** Recompute pair ordering on the exact evaluation subset, verify the intervention sign and token position, and compare representations under raw-sentence encoding versus the same chat-template context used during generation. Also record the activation shift at layer 11 itself and at later layers.
6. **Then test localization.** If layer 11 remains competitive under independent evaluation, steer selected token positions instead of every generated position and compare mean-pooled versus final-token directions. This may retain the stylistic benefit while reducing content drift.
